# Guest operations brief: Walt Disney World wait times

This notebook is the written briefing behind the Streamlit app. Three questions, same data:

1. How much do **posted** waits overshoot **actual** waits?
2. What is the **shape of a park day** for headliners in Orlando?
3. Can a transparent baseline tell us whether a live wait is **hot or cold**?

Not affiliated with The Walt Disney Company. Historical waits: TouringPlans.com. Live waits: Powered by ThemeParks.wiki.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from wdw.config import SAMPLE_HOURLY_PARQUET
from wdw.model import expected_wait, posted_vs_actual
from wdw.typical import typical_day_curve
from wdw.warehouse import load_hourly

hourly = load_hourly()
hourly["observed_at"] = pd.to_datetime(hourly["observed_at"])
print(f"Rows: {len(hourly):,}  Attractions: {hourly['attraction_name'].nunique()}  "
      f"Dates: {hourly['park_date'].min()} → {hourly['park_date'].max()}")
if SAMPLE_HOURLY_PARQUET.exists() and len(hourly) < 20000:
    print("Using the compact sample warehouse. Run wdw-ingest-history and wdw-build for the full series.")

## 1. Posted vs actual wait

Posted standby is what the park prints on a sign. Actual wait is what a guest reported standing in line. A positive gap means the sign was conservative — a guest-communication buffer, not just a forecasting error.

TouringPlans and follow-on academic work have found Disney often over-posts. We quantify that buffer by attraction and by hour.

In [ ]:
summary = posted_vs_actual(hourly)
display(summary.round(1))

print(f"Overall median posted − actual: {hourly['posted_minus_actual'].median():.1f} minutes")

fig = px.bar(
    summary.sort_values("bias_mean"),
    x="bias_mean",
    y="attraction_name",
    color="park_name",
    labels={
        "bias_mean": "Mean posted minus actual (minutes)",
        "attraction_name": "Attraction",
        "park_name": "Park",
    },
    title="Posted-wait buffer by attraction",
)
fig.update_layout(paper_bgcolor="white")
fig.show()

by_hour = (
    hourly.dropna(subset=["posted_minus_actual"])
    .groupby("hour", observed=True)["posted_minus_actual"]
    .median()
    .reset_index()
)
fig_h = px.line(
    by_hour,
    x="hour",
    y="posted_minus_actual",
    markers=True,
    labels={
        "hour": "Hour of day (local)",
        "posted_minus_actual": "Median posted − actual (minutes)",
    },
    title="Posted-wait buffer by hour of day",
)
fig_h.show()

**Finding.** Posted waits sit above actual waits across the headliner set. The buffer is larger on constrained-capacity E-tickets than on high-throughput dark rides. For a guest-experience team, that is the cost of not wanting guests to feel the sign lied.

## 2. Shape of a park day

Rope drop, midday, evening. Early entry (historically Extra Magic Hours) pulls demand into the first two hours. We use median posted wait by hour, with a 25th–75th band.

In [ ]:
focus = "Seven Dwarfs Mine Train"
key = hourly.loc[hourly["attraction_name"] == focus, "attraction_key"].iloc[0]
curve = typical_day_curve(hourly, key)

fig = go.Figure()
fig.add_trace(go.Scatter(x=curve["hour"], y=curve["posted_p75"], line=dict(width=0), showlegend=False))
fig.add_trace(
    go.Scatter(
        x=curve["hour"],
        y=curve["posted_p25"],
        fill="tonexty",
        name="25th–75th percentile",
    )
)
fig.add_trace(go.Scatter(x=curve["hour"], y=curve["posted_median"], name="Median posted"))
fig.add_trace(go.Scatter(x=curve["hour"], y=curve["actual_median"], name="Median actual", line=dict(dash="dash")))
fig.update_layout(
    title=f"Typical day · {focus}",
    xaxis_title="Hour of day (local)",
    yaxis_title="Wait (minutes)",
    xaxis=dict(dtick=1),
)
fig.show()

all_curves = typical_day_curve(hourly)
fig2 = px.line(
    all_curves,
    x="hour",
    y="posted_median",
    color="attraction_name",
    labels={
        "hour": "Hour of day (local)",
        "posted_median": "Median posted wait (minutes)",
        "attraction_name": "Attraction",
    },
    title="Typical-day posted waits, TouringPlans headliners",
)
fig2.show()

if "early_entry" in hourly.columns:
    ee = (
        hourly.dropna(subset=["posted_wait_median"])
        .groupby(["early_entry", "hour"], observed=True)["posted_wait_median"]
        .median()
        .reset_index()
    )
    ee["early_entry"] = ee["early_entry"].map({0: "Standard opening", 1: "Early entry / EMH morning"})
    fig3 = px.line(
        ee,
        x="hour",
        y="posted_wait_median",
        color="early_entry",
        labels={"hour": "Hour of day (local)", "posted_wait_median": "Median posted wait (minutes)", "early_entry": "Day type"},
        title="Early entry shifts demand into the morning",
    )
    fig3.show()

**Finding.** The cheapest hour is still the first hour. Headliners peak late morning through mid-afternoon. Early-entry mornings are not a free lunch — they move the queue earlier rather than deleting it.

## 3. Now vs expected

Expected wait is the TouringPlans historical median for the same attraction, hour, and weekday. Live standby minus that baseline flags a hot hour.

No trained model. We are asking whether this hour matches the park's own history.

In [ ]:
baseline = (
    hourly.groupby(["attraction_name", "hour", "weekday"], observed=True)["posted_wait_median"]
    .median()
    .rename("expected_wait")
    .reset_index()
)
noon_saturday = baseline.loc[(baseline["hour"] == 12) & (baseline["weekday"] == 5)].sort_values("expected_wait")
display(noon_saturday.round(1))

print("Expected wait on Mission control is this median for the current Eastern hour and weekday.")
print("Live ThemeParks.wiki standby minus expected_wait is delta_vs_expected.")

**Finding.** Hour and attraction do most of the work. If live standby is well above this hour's historical median, the park is running hot — not that a model "won."

## Caveats

- TouringPlans coverage is a headliner subset, not every attraction.
- Actual waits are guest-reported and sparse compared with posted waits.
- Splash Mountain historical waits are not Tiana's Bayou Adventure.
- Live ThemeParks.wiki data is third-party, cached ~5 minutes, and not an official Disney feed.
- If I had production data at Disney, next I would add virtual-queue abandon rates, Lightning Lane conversion, weather at the park, and special-ticket events.